# Check General Noise
We see if the General Noise is created consistently throughout the runs, and print its Kraus Operators

In [57]:
using CSV
using DataFrames
using TOML
using JLD2
using Random

ROOT_DIR = joinpath("..")
EXPERIMENTS_DIR = joinpath(ROOT_DIR, "experiments")
SUMMARY_CSV = joinpath(EXPERIMENTS_DIR, "iterations_summary.csv")

include(joinpath(ROOT_DIR, "julia", "src", "utils.jl"))
include(joinpath(ROOT_DIR, "julia", "src", "operators.jl"))

get_kraus_from_map

In [60]:
df = CSV.read(SUMMARY_CSV, DataFrame)
df = filter(row -> !occursin("20261106", row.name), df)
df = filter(row -> occursin("rand", row.name), df)
unique(df.seed)

1-element Vector{Int64}:
 1214

In [65]:
baseline_experiment = "1qbt_rand_rand_auto"
cfg = TOML.parsefile(joinpath(EXPERIMENTS_DIR, baseline_experiment, "config.toml"))
recovery_operators = JLD2.load(joinpath(EXPERIMENTS_DIR, baseline_experiment, "data", "superoperators.jld2"))

kraus = recovery_operators["noise_options_kraus"][1]
for k in eachindex(kraus)
  display(kraus[k])
end

2×2 Matrix{ComplexF64}:
   0.339608-0.0981709im  -0.576111+0.122395im
 -0.0976442+0.246524im   0.0367269-0.400818im

2×2 Matrix{ComplexF64}:
 -0.120724+0.249831im   -0.297298-0.00511733im
  0.030419-0.0331122im   0.129726-0.105732im

2×2 Matrix{ComplexF64}:
 -0.0683689+0.275255im  0.0476068+0.119511im
  -0.216772-0.623988im  -0.341589-0.278275im

2×2 Matrix{ComplexF64}:
 -0.0941674-0.22764im   0.297009+0.200167im
   0.201005+0.32837im  0.0111454+0.188723im

In [80]:
test_experiment = df[1, :name]
cfg_test = TOML.parsefile(joinpath(EXPERIMENTS_DIR, test_experiment, "config.toml"))
recovery_operators_test = JLD2.load(joinpath(EXPERIMENTS_DIR, test_experiment, "data", "superoperators.jld2"))

display(cfg_test)

kraus_test = recovery_operators_test["noise_options_kraus"][1]
expand_kraus_operators(kraus, cfg_test["n_qubits"])[1]

Dict{String, Any} with 16 entries:
  "n_timesteps"       => 5
  "starting_state"    => "thermal"
  "dt"                => 0.0314159
  "noise_options"     => Any[Any[0.5, "general", 20.0], Any[0.5, "amplitude_dam…
  "name"              => "20261306_rand_amp_n2_b01"
  "correlated_noise"  => false
  "plots"             => Dict{String, Any}("choices"=>true, "plot_title"=>"code…
  "n_states"          => 100
  "ancilla_alpha"     => 0.5
  "recovery_type"     => "codespace"
  "collision_unitary" => "jc"
  "n_qubits"          => 2
  "beta"              => 0.1
  "real_noise"        => "general"
  "seed"              => 1214
  "ancilla_state"     => "ground_qubit"

4×4 Matrix{ComplexF64}:
    0.105696-0.0666793im  -0.183636+0.0981237im  …   0.316923-0.141026im
 -0.00895925+0.0933074im  -0.026876-0.139727im      0.0278994+0.235411im
 -0.00895925+0.0933074im  0.0260806-0.153977im      0.0278994+0.235411im
  -0.0512398-0.0481433im  0.0952253+0.0481917im     -0.159306-0.0294416im

In [78]:
kraus_test[1]

4×4 Matrix{ComplexF64}:
  0.0888986+0.227611im   -0.0121687+0.0184928im   …  -0.00200433+6.90778e-5im
  0.0991453-0.0285772im    0.180344-0.122972im         0.0185358+0.00689007im
  0.0991453-0.0285772im  0.00730528+0.00583201im       0.0185358+0.00689007im
 -0.0081816-0.0427944im  -0.0447328-0.0805879im        -0.143169-0.132371im

In [62]:
recovery_operators = Dict{String, Any}()
i = 0
for experiment in eachrow(df)
  experiment_dir = joinpath(EXPERIMENTS_DIR, String(experiment.name))
  config_path = joinpath(experiment_dir, "config.toml")
  cfg = TOML.parsefile(config_path)
  n_qubits = cfg["n_qubits"]
  corr_noise = cfg["correlated_noise"]

  noises = [option[2] for option in cfg["noise_options"]]
  if all(noise -> noise != "general", noises)
    println("Experiment $(experiment.name) has unexpected noise configuration; skipping.")
    continue
  end

  if cfg["seed"] != 1214
    continue
  end
  
  general_noise_idx = findfirst(noise -> noise == "general", noises)
  general_noise = noises[general_noise_idx]

  recovery_operators = load(joinpath(experiment_dir, "data", "superoperators.jld2"))
  # Check if the extended Kraus operators coincide with the baseline
  experiment_kraus = recovery_operators["noise_options_kraus"][general_noise_idx]
  extended_baseline_kraus = 
    corr_noise && n_qubits > 1 ? correlate_kraus_operators(kraus, n_qubits) :
    expand_kraus_operators(kraus, n_qubits)
  if length(experiment_kraus) != length(extended_baseline_kraus)
    println("Experiment $(experiment.name) has a different number of Kraus operators than the baseline; skipping.")
    continue
  end
  all_close = true
  for (k1, k2) in zip(experiment_kraus, extended_baseline_kraus)
    if !isapprox(k1, k2, atol=1e-8)
      all_close = false
      break
    end
  end
  if all_close
    println("Experiment $(experiment.name) has Kraus operators that coincide with the baseline.")
  else
    println("Experiment $(experiment.name) has Kraus operators that differ from the baseline.")
  end 


  i += 1
  if i >= 50
    break
  end
end


Experiment 20261306_rand_amp_n2_b01 has Kraus operators that differ from the baseline.
Experiment 20261306_rand_amp_n2_b05 has Kraus operators that differ from the baseline.
Experiment 20261306_rand_amp_n2_b10 has Kraus operators that differ from the baseline.
Experiment 20261306_rand_amp_n2_b15 has Kraus operators that differ from the baseline.
Experiment 20261306_rand_amp_n2_b20 has Kraus operators that differ from the baseline.
Experiment 20261306_rand_amp_n2_b25 has Kraus operators that differ from the baseline.
Experiment 20261306_rand_amp_n3_b01 has Kraus operators that differ from the baseline.
Experiment 20261306_rand_amp_n3_b05 has Kraus operators that differ from the baseline.
Experiment 20261306_rand_amp_n2_b01 has Kraus operators that differ from the baseline.
Experiment 20261306_rand_amp_n2_b05 has Kraus operators that differ from the baseline.
Experiment 20261306_rand_amp_n2_b10 has Kraus operators that differ from the baseline.
Experiment 20261306_rand_amp_n2_b15 has Kra

In [ ]:

recovery_operators["noise_options_kraus"][1][1]

8×8 Matrix{ComplexF64}:
 -0.0856362-0.121285im   -0.0255685+0.135462im    …   -0.43596+0.0624705im
  0.0213336+0.0594544im   -0.130976+0.0301165im      -0.066743-0.235857im
  0.0213336+0.0594544im   0.0244921-0.141376im       -0.066743-0.235857im
 -0.0622148-0.0732091im    0.058662-0.00339548im      0.163808-0.0819297im
  0.0213336+0.0594544im   0.0244921-0.141376im       -0.066743-0.235857im
 -0.0622148-0.0732091im    0.058662-0.00339548im  …   0.163808-0.0819297im
 -0.0622148-0.0732091im   0.0798042+0.152218im        0.163808-0.0819297im
   0.108097-0.0043766im  -0.0782742+0.0188037im      0.0724183+0.145738im